# L10 — Sensor Data, Jupyter Notebooks, and Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L10/L10_starter.ipynb)

**Goal:** load a recording captured from a micro:bit accelerometer, plot it, compute magnitude, smooth it, and find the peaks (jumps).

**Where to run this:** Google Colab (no install). Click the badge above, or *File → Open notebook → GitHub* in Colab. Runs locally too if you have JupyterLab — see the README in the course repo.

**How to use this notebook:** press **`Shift + Enter`** on each cell, from top to bottom. Read the words between cells — they explain what each step does. You'll see the same five steps twice: once on a sample recording, then again on your own.

## Step 0 — Imports and the CSV loader

`load_microbit_csv(...)` hides the quirks of the MakeCode CSV download (tab-separated, one timestamp column per axis) and returns a clean table with columns `t`, `x`, `y`, `z`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def load_microbit_csv(path_or_url):
    """Read a micro:bit sensor CSV and return a DataFrame with columns t, x, y, z."""
    if str(path_or_url).startswith(("http://", "https://")):
        import urllib.request
        with urllib.request.urlopen(path_or_url) as r:
            first = r.readline().decode("utf-8", errors="ignore").rstrip("\r\n")
    else:
        with open(path_or_url) as f:
            first = f.readline().rstrip("\r\n")

    if first.startswith("sep="):                # MakeCode download: 'sep=\t' hint on line 1
        sep = first.split("=", 1)[1] or "\t"
        df = pd.read_csv(path_or_url, sep=sep, skiprows=1)
    else:
        df = pd.read_csv(path_or_url)
        if df.shape[1] == 1:                    # looked like CSV but was actually TSV
            df = pd.read_csv(path_or_url, sep="\t")

    # MakeCode repeats the time column once per axis — drop the duplicates.
    keep, seen_time = [], False
    for c in df.columns:
        is_time = c.lower().startswith("time (") or c == "t"
        if is_time and seen_time:
            continue
        if is_time:
            seen_time = True
        keep.append(c)
    df = df[keep].copy()
    time_cols = [c for c in df.columns if c.lower().startswith("time (") or c == "t"]
    if time_cols:
        df = df.rename(columns={time_cols[0]: "t"})
    return df[["t", "x", "y", "z"]]


print("running in Colab" if IN_COLAB else "running locally")

# Part 1 — Explore the sample recording

We'll start with a bundled walking trace so you can see the whole pipeline before capturing your own data. The sample loads straight from the public course repo — no upload needed.

## Step 1 — Load the sample CSV

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-walk.csv"

if IN_COLAB:
    df = load_microbit_csv(SAMPLE_URL)
else:
    df = load_microbit_csv("../data/sample-walk.csv")

print(df.shape)
df.head()

**Sanity check:** the columns should be `t`, `x`, `y`, `z`. `t` is the timestamp in seconds.

## Step 2 — Plot X, Y, Z over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["t"], df["x"], label="x")
ax.plot(df["t"], df["y"], label="y")
ax.plot(df["t"], df["z"], label="z")
ax.set_xlabel("time (s)")
ax.set_ylabel("acceleration (mg)")
ax.legend()
ax.grid(True)
plt.show()

## Step 3 — Compute magnitude

$|a| = \sqrt{x^2 + y^2 + z^2}$. At rest this is ~1000 mg (gravity).

In [ ]:
df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag"], color="purple")
ax.axhline(1000, linestyle="--", color="gray", label="rest")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 4 — Smooth the signal

A 5-sample rolling mean is enough to kill most accelerometer noise without blurring real motion.

In [ ]:
df["mag_smooth"] = df["mag"].rolling(window=5, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag"],        alpha=0.3, label="raw")
ax.plot(df["t"], df["mag_smooth"],            label="smoothed")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 5 — Find the peaks

`height` is the minimum peak value (in mg). `distance` is the minimum number of samples between peaks. The red `x`'s mark the detected peaks.

In [ ]:
peaks, _ = find_peaks(df["mag_smooth"].fillna(0),
                      height=1300,
                      distance=4)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag_smooth"])
ax.plot(df["t"].iloc[peaks], df["mag_smooth"].iloc[peaks], "rx", markersize=12)
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
plt.show()

print(f"Detected {len(peaks)} peaks")

# Part 2 — Now do the same on *your* recording 🎚️

1. **Capture** a 30-second recording in MakeCode (Chrome or Edge): *Show Console Device → Download*. See the L10 slides for details.
2. **Upload** the CSV with the cell below — this replaces the sample `df` with your data.
3. **Keep pressing `Shift + Enter`**. The five steps below are the same as Part 1, but running on *your* recording. You don't need to write any code — just look at the plots.
4. When you reach the Peaks cell, come back and change `height` and `distance` until the red x's line up with what you actually did.
5. **Save your work:** *File → Download .ipynb* before the runtime times out.

## Step 1 (your recording) — Upload and load the CSV

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()        # opens a file picker
    MY_FILE = next(iter(uploaded))
else:
    MY_FILE = "../data/microbit-data-2026-04-17T20-52-48-462Z.csv"

df = load_microbit_csv(MY_FILE)
print(df.shape)
df.head()

## Step 2 (your recording) — Plot X, Y, Z over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["t"], df["x"], label="x")
ax.plot(df["t"], df["y"], label="y")
ax.plot(df["t"], df["z"], label="z")
ax.set_xlabel("time (s)")
ax.set_ylabel("acceleration (mg)")
ax.legend()
ax.grid(True)
plt.show()

## Step 3 (your recording) — Compute magnitude

In [ ]:
df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag"], color="purple")
ax.axhline(1000, linestyle="--", color="gray", label="rest")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 4 (your recording) — Smooth the signal

In [ ]:
df["mag_smooth"] = df["mag"].rolling(window=5, center=True).mean()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag"],        alpha=0.3, label="raw")
ax.plot(df["t"], df["mag_smooth"],            label="smoothed")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 5 (your recording) — Find the peaks

Change `height` and `distance` below and re-run this cell (`Shift + Enter`) until the red x's land on the moves you actually did.

In [ ]:
peaks, _ = find_peaks(df["mag_smooth"].fillna(0),
                      height=1300,
                      distance=4)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag_smooth"])
ax.plot(df["t"].iloc[peaks], df["mag_smooth"].iloc[peaks], "rx", markersize=12)
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
plt.show()

print(f"Detected {len(peaks)} peaks")

## Reflect

**Write 2–3 sentences below** describing what feature distinguishes your move from a baseline (rest or walking). For example: *the peaks are taller*, *there are more of them per second*, *the magnitude never returns to 1000 mg between moves*, etc.

*Your answer here.*